# Guia Completo de Bancos de Dados com SQLAlchemy
### Referência Prática Organizada em Três Níveis de Complexidade

---
> **SQLite local:** usaremos um banco em arquivo (`loja.db`) — não é preciso instalar servidor. Trocar a URL do `engine` adapta o mesmo código para PostgreSQL/MySQL.

## Sumário

| Módulo | Nível | Temas |
|--------|-------|-------|
| **1** | Básico | Engine, SQL puro, Parâmetros Seguros, pandas |
| **2** | Intermediário | Core: Tabelas, insert, select, update, delete, agregações |
| **3** | Avançado | ORM: Classes, Sessões, Relacionamentos, Joins |

---

## Configuração do Ambiente

Execute a célula abaixo: apaga o banco anterior (se existir), cria o `engine` do SQLite e importa as bibliotecas.

Instalação no terminal (uma única vez): `pip install sqlalchemy`

> **Reexecutar todo o notebook sempre reinicia os dados** — a célula de setup apaga o `loja.db`.

In [ ]:
import os
import pandas as pd

from sqlalchemy import (Column, Float, Integer, MetaData, String, Table,
                        and_, or_, create_engine, delete, func, insert,
                        select, text, update)

# Reinicia o banco: remove o arquivo anterior, se existir
if os.path.exists('loja.db'):
    os.remove('loja.db')

# SQLite em arquivo — sem necessidade de servidor
engine = create_engine('sqlite:///loja.db')

print('Engine pronto. Banco:', engine.url.database)

# MÓDULO 1 — Nível Básico
## Engine, SQL Puro, Parâmetros Seguros e Pandas

---
## 1.1 O Que é o SQLAlchemy

Biblioteca Python que conecta aplicações a bancos de dados SQL. Oferece duas camadas de trabalho:

| Camada | Ideia | Quando usar |
|--------|-------|-------------|
| **Core** | SQL como **expressões Python** (`select`, `insert`...) | Consultas dinâmicas, scripts de dados |
| **ORM** | Tabelas viram **classes** e linhas viram **objetos** | Aplicações com modelos de domínio |

Peças fundamentais: **Engine** (conexão ao banco), **Connection** (executa comandos), **Result** (linhas retornadas).

---
## 1.2 Executando SQL Puro: `text()` e `engine.begin()`

Para instruções com SQL literal, envolva a string em **`text(...)`**. O bloco `with engine.begin()` abre uma **transação** que é confirmada automaticamente ao final.

**Sintaxe:** `with engine.begin() as conn: conn.execute(text('SQL...'))`

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE vendas (
            id INTEGER PRIMARY KEY,
            cliente_id INTEGER,
            valor REAL,
            estado TEXT,
            categoria TEXT,
            status TEXT
        )
    '''))

print('Tabela vendas criada!')

---
## 1.3 Parâmetros Seguros: Use `:nome`, NUNCA f-strings

Valores **nunca devem ser colados** no SQL (risco de **injeção SQL**). Use placeholders `:nome` e passe um dicionário (ou lista de dicionários).

**Sintaxe:** `conn.execute(text('... = :uf'), {'uf': 'SP'})`

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        INSERT INTO vendas (cliente_id, valor, estado, categoria, status)
        VALUES (:cliente, :valor, :estado, :categoria, :status)
    '''), [
        {'cliente': 101, 'valor': 3500.75, 'estado': 'SP', 'categoria': 'Informática', 'status': 'Concluído'},
        {'cliente': 102, 'valor': 189.50, 'estado': 'RJ', 'categoria': 'Acessórios', 'status': 'Pendente'},
    ])

print('2 registros inseridos')

---
## 1.4 Lendo Resultados

`connection.execute()` devolve um objeto **`Result`**. Use **`.mappings().all()`** para obter dicionários prontos para o pandas.

**Sintaxe:** `resultado.mappings().all()` → lista de dicts → `pd.DataFrame(...)`

In [ ]:
with engine.connect() as conn:
    resultado = conn.execute(text('SELECT * FROM vendas'))
    linhas = resultado.mappings().all()

print('linhas:', linhas)

df_vendas = pd.DataFrame(linhas)
df_vendas

In [ ]:
# Atalho: pd.read_sql_query já devolve um DataFrame pronto
df_vendas2 = pd.read_sql_query('SELECT * FROM vendas', engine)
df_vendas2

---
## 1.5 Por Que Parâmetros? Exemplo de Injeção SQL

Se montássemos o SQL com f-string, um valor malicioso como `SP' OR 1=1 --` **mudaria a lógica da consulta**. Com parâmetros, ele é tratado apenas como dado — e a busca não encontra nada.

In [ ]:
valor_malicioso = "SP' OR 1=1 --"

# Como NÃO fazer (injeção SQL):
# sql_perigoso = f"SELECT * FROM vendas WHERE estado = '{valor_malicioso}'"

# Forma segura:
with engine.connect() as conn:
    resultado = conn.execute(
        text('SELECT * FROM vendas WHERE estado = :uf'),
        {'uf': valor_malicioso}
    )
    linhas = resultado.mappings().all()

print('Linhas retornadas com valor malicioso (param seguro):', len(linhas))

# MÓDULO 2 — Nível Intermediário
## Core: Tabelas, Insert, Select, Update, Delete e Agregações

---
## 2.1 Definindo uma Tabela: `Table` + `MetaData`

No **Core**, a tabela é descrita programaticamente. `metadata.create_all(engine)` cria no banco as tabelas que ainda não existem.

**Sintaxe:** `tabela = Table('nome', metadata, Column('col', Tipo))`

In [ ]:
metadata = MetaData()

clientes = Table(
    'clientes', metadata,
    Column('id', Integer, primary_key=True),
    Column('nome', String(60)),
    Column('uf', String(2)),
    Column('ativo', Integer),
)

metadata.create_all(engine)
print('Tabela clientes criada')

---
## 2.2 Inserindo em Lote: `insert`

`insert(tabela)` monta o comando. Passar uma **lista de dicionários** insere várias linhas de uma vez.

**Sintaxe:** `conn.execute(insert(tabela), [lista_de_dicts])`

In [ ]:
registros = [
    {'nome': 'Ana Souza',   'uf': 'SP', 'ativo': 1},
    {'nome': 'Bruno Lima',  'uf': 'MG', 'ativo': 1},
    {'nome': 'Carla Melo',  'uf': 'RJ', 'ativo': 0},
    {'nome': 'Diego Prado', 'uf': 'SC', 'ativo': 1},
]

with engine.begin() as conn:
    conn.execute(insert(clientes), registros)

print('4 clientes inseridos')

---
## 2.3 Consultando: `select` com `where`, `order_by`, `limit`

O `select` encadeia a construção da consulta. As colunas são acessadas com **`tabela.c.coluna`** (o `c` vem de *columns*).

**Sintaxe:** `select(tabela).where(tabela.c.uf == 'SP')`

In [ ]:
stmt = select(clientes).where(clientes.c.uf == 'SP')

with engine.connect() as conn:
    resultado = conn.execute(stmt).mappings().all()

pd.DataFrame(resultado)

In [ ]:
stmt = (select(clientes.c.nome, clientes.c.uf)
        .where(and_(clientes.c.ativo == 1, clientes.c.uf != 'RJ'))
        .order_by(clientes.c.nome)
        .limit(10))

with engine.connect() as conn:
    resultado = conn.execute(stmt).mappings().all()

print(resultado)

---
## 2.4 Atualizando e Excluindo

- `update(tabela).where(cond).values(col=tipo)` — altera linhas.
- `delete(tabela).where(cond)` — remove linhas.

**Sintaxe:** `conn.execute(update(tabela).where(...).values(...))`

In [ ]:
with engine.begin() as conn:
    conn.execute(update(clientes).where(clientes.c.nome == 'Carla Melo').values(ativo=1))
    conn.execute(delete(clientes).where(clientes.c.nome == 'Diego Prado'))

# Conferindo o estado atual da tabela
pd.read_sql_query('SELECT * FROM clientes', engine)

---
## 2.5 Funções Agregadas: `func` + `group_by`

`func.count()`, `func.sum()`, `func.avg()` reproduzem as funções SQL. O **`.label('nome')`** dá nome à coluna calculada.

**Sintaxe:** `select(tabela.c.uf, func.count().label('total')).group_by(tabela.c.uf)`

In [ ]:
stmt = (select(clientes.c.uf, func.count().label('total'))
        .group_by(clientes.c.uf))

pd.read_sql_query(stmt, engine)

---
## 2.6 Integração com Pandas: `to_sql`

`DataFrame.to_sql(nome, engine)` **cria a tabela e grava os dados** de uma vez — o caminho mais rápido para popular um banco a partir de arquivos.

In [ ]:
df = pd.read_csv('vendas.csv', sep=';', encoding='utf-8')

# Seleciona as colunas de interesse
dados = df[['cliente_id', 'valor', 'estado', 'categoria', 'status']]
dados.to_sql('vendas_completo', engine, index=False)

print('Registros carregados no banco:', len(dados))

In [ ]:
# Perguntando algo ao banco: qual estado tem mais vendas?
consulta = '''
SELECT estado,
       COUNT(*)            AS total,
       ROUND(AVG(valor), 2) AS media
FROM vendas_completo
GROUP BY estado
ORDER BY total DESC
'''

pd.read_sql_query(consulta, engine)

# MÓDULO 3 — Nível Avançado
## ORM: Classes, Sessões, Relacionamentos e Joins

---
## 3.1 Mapeando Tabelas com Classes

No **ORM**, cada tabela vira uma **classe** e cada linha vira um **objeto**. Usamos `declarative_base()` e `mapped_column` (estilo 2.0).

**Sintaxe:** `class Cliente(Base): __tablename__ = 'tabela'; id = mapped_column(Integer, primary_key=True)`

In [ ]:
from sqlalchemy import ForeignKey
from sqlalchemy.orm import declarative_base, mapped_column, relationship, sessionmaker

Base = declarative_base()


class Cliente(Base):
    """Um cliente pode ter vários pedidos."""
    __tablename__ = 'clientes_orm'

    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String(60))
    uf = mapped_column(String(2))

    pedidos = relationship('Pedido', back_populates='cliente')


class Pedido(Base):
    """Cada pedido pertence a um único cliente."""
    __tablename__ = 'pedidos_orm'

    id = mapped_column(Integer, primary_key=True)
    descricao = mapped_column(String(100))
    valor = mapped_column(Float)
    cliente_id = mapped_column(ForeignKey('clientes_orm.id'))

    cliente = relationship('Cliente', back_populates='pedidos')


Base.metadata.create_all(engine)
print('Tabelas ORM criadas:', list(Base.metadata.tables.keys()))

---
## 3.2 Sessões: Salvando Objetos

A **`Session`** é a ponte entre os objetos Python e o banco. `add` / `add_all` marcam os objetos e **`commit()`** grava tudo de uma vez.

**Sintaxe:** `sessao.add(objeto); sessao.commit()`

In [ ]:
Session = sessionmaker(bind=engine)
sessao = Session()

ana = Cliente(nome='Ana Souza', uf='SP')
bruno = Cliente(nome='Bruno Lima', uf='MG')

sessao.add_all([ana, bruno])
sessao.commit()   # persiste no banco

print('Clientes salvos com IDs:', ana.id, 'e', bruno.id)

---
## 3.3 Consultando com `select` e `scalars()`

Com ORM, `sessao.execute(select(Cliente).where(...))` retorna *linhas*; **`.scalars().all()`** devolve os **objetos** da classe.

In [ ]:
clientes_orm = sessao.execute(select(Cliente)).scalars().all()

for c in clientes_orm:
    print(c.id, '-', c.nome, '(', c.uf, ')')

In [ ]:
sp = sessao.execute(select(Cliente).where(Cliente.uf == 'SP')).scalars().all()
print('Clientes de SP:', [c.nome for c in sp])

---
## 3.4 Relacionamentos: Criando Pedidos

Com `relationship`, navegamos de **objeto para objeto**. Ao criar um `Pedido` com `cliente=ana`, o ORM preenche a chave estrangeira automaticamente.

In [ ]:
ana = sessao.execute(select(Cliente).where(Cliente.nome == 'Ana Souza')).scalar_one()

pedido1 = Pedido(descricao='Laptop', valor=4500.0, cliente=ana)
pedido2 = Pedido(descricao='Monitor 27"', valor=1200.0, cliente=ana)

sessao.add_all([pedido1, pedido2])
sessao.commit()

print('Pedidos da Ana:', [(p.descricao, p.valor) for p in ana.pedidos])

---
## 3.5 Join entre Classes

Além do `relationship`, dá para fazer o **join clássico** por chave estrangeira e devolver o resultado direto para um DataFrame.

In [ ]:
stmt_join = (select(Cliente.nome, Pedido.descricao, Pedido.valor)
             .join(Pedido, Pedido.cliente_id == Cliente.id))

pd.read_sql_query(stmt_join, engine)

In [ ]:
# Total gasto por cliente (agregação no ORM)
stmt_total = (select(Cliente.nome, func.sum(Pedido.valor).label('total_gasto'))
              .join(Pedido, Pedido.cliente_id == Cliente.id)
              .group_by(Cliente.nome))

pd.read_sql_query(stmt_total, engine)

---
## Fechando a Sessão

Bons hábitos: fechar a sessão (e eventualmente o engine) após o uso.

In [ ]:
sessao.close()
print('Sessão fechada. Exemplos concluídos!')

---
# Resumo Rápido de Referência

| Função | Descrição |
|--------|-----------|
| `create_engine('sqlite:///arquivo.db')` | Conecta ao banco (aceita postgresql/mysql) |
| `engine.begin()` | Transação com commit automático |
| `engine.connect()` | Conexão de leitura |
| `text('SQL ...')` | SQL literal |
| `:param` + dicionário | Parâmetros seguros (anti injeção SQL) |
| `.mappings().all()` | Resultado como lista de dicionários |
| `pd.read_sql_query(sql, engine)` | SQL/select direto para DataFrame |
| `DataFrame.to_sql('tabela', engine)` | Grava o DataFrame no banco |
| `Table('t', metadata, Column(...))` | Define tabela no Core |
| `select` / `insert` / `update` / `delete` | Comandos do Core |
| `tabela.c.coluna` | Acesso a colunas no Core |
| `and_()` / `or_()` | Combinação de condições |
| `func.count()` / `func.sum()` | Funções agregadas SQL |
| `declarative_base()` / `mapped_column` | Tabelas como classes (ORM) |
| `sessionmaker(bind=engine)` | Fábrica de sessões |
| `sessao.add()` / `.commit()` | Persistir objetos |
| `.scalars().all()` | Objetos de uma consulta ORM |
| `relationship` / `ForeignKey` | Relacionamento entre tabelas |